# Additional analyses -- classification, enrichment, SHAP, CatBoost, ablation

**Self-contained.** Put this notebook in the same folder as `ki_verified_5-HT2A.csv`,
`ki_verified_Dopamine_D3.csv` and `curated_15desc.csv` (or a `data/` subfolder), then
**Run All**. Everything here is *additive* to the thesis regression work -- nothing is removed.

Thresholds: pKi = 9 - log10(Ki[nM]).  100 nM -> pKi 7.0 (primary, balanced);  1 uM -> pKi 6.0.

In [ ]:
!pip -q install rdkit xgboost catboost scikit-learn pandas matplotlib descriptastorus

In [ ]:
import os, numpy as np, pandas as pd, warnings, time; warnings.filterwarnings('ignore')
from rdkit import Chem, DataStructs, RDLogger; RDLogger.DisableLog('rdApp.*')
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.Scaffolds import MurckoScaffold
from descriptastorus.descriptors import rdNormalizedDescriptors
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

LOCKED = 'data/curated_15desc.csv' if os.path.exists('data/curated_15desc.csv') else 'curated_15desc.csv'
FILES  = {'5-HT2A': 'ki_verified_5-HT2A.csv', 'Dopamine D3': 'ki_verified_Dopamine_D3.csv'}
KEY    = {'5-HT2A': '5-HT2A', 'Dopamine D3': 'D3'}
for f in [LOCKED, *FILES.values()]:
    assert os.path.exists(f), f'MISSING FILE: {f}'
print('all input files found')

In [ ]:
# ---- curation + build the modelling sets (same pipeline as the thesis) ----
sr = SaltRemover(); pep = Chem.MolFromSmarts('[NX3][CX4][CX3](=O)[NX3][CX4][CX3](=O)')
def parent(s):
    m = Chem.MolFromSmiles(str(s)) if isinstance(s, str) else None
    if m is None: return None
    p = sr.StripMol(m, dontRemoveEverything=True)
    if p is None or p.GetNumAtoms() == 0:
        fr = Chem.GetMolFrags(m, asMols=True); p = max(fr, key=lambda x: x.GetNumAtoms()) if fr else m
    return p
def landrum(g):
    v = g['pKi'].round(2); v = v[~v.duplicated(keep='first')]
    v = [x for x in v if not any(abs(x-o-3.0) < 1e-6 or abs(x-o+3.0) < 1e-6 for o in v)]
    return max(v) if v else np.nan
locked = pd.read_csv(LOCKED)
def build(name):
    raw = pd.read_csv(FILES[name]); raw['standard_value'] = pd.to_numeric(raw['standard_value'], errors='coerce')
    raw = raw[raw['standard_value'] > 0].copy(); raw['pKi'] = 9.0 - np.log10(raw['standard_value'])
    raw['mol'] = raw['canonical_smiles'].apply(parent); raw = raw[raw['mol'].notna()]
    raw = raw[raw['mol'].apply(lambda m: Descriptors.MolWt(m)) < 700]
    raw = raw[~raw['mol'].apply(lambda m: m.HasSubstructMatch(pep))]
    raw['parent_smiles'] = raw['mol'].apply(Chem.MolToSmiles)
    agg = raw.groupby('molecule_chembl_id').apply(
            lambda g: pd.Series({'pKi': landrum(g), 'parent_smiles': g['parent_smiles'].iloc[0]})
          ).reset_index().dropna(subset=['pKi'])
    ex = locked[locked.target.str.contains(KEY[name], case=False)][['molecule_chembl_id','pKi','parent_smiles']].drop_duplicates('molecule_chembl_id')
    pr = agg[~agg.molecule_chembl_id.isin(ex.molecule_chembl_id)]
    both = pd.concat([ex, pr], ignore_index=True).drop_duplicates('molecule_chembl_id').reset_index(drop=True)
    print(f'{name}: {len(ex)} clinical + {len(pr)} new = {len(both)} compounds')
    return both
sets = {n: build(n) for n in FILES}

In [ ]:
# ---- featurise: ECFP4 (2048) + DescriptaStorus RDKit2DNormalized (~200) = fused ----
def featurize(dfset):
    gen = rdNormalizedDescriptors.RDKit2DNormalized()
    mols = [Chem.MolFromSmiles(s) for s in dfset.parent_smiles]; keep = [i for i, m in enumerate(mols) if m]
    dfset = dfset.iloc[keep].reset_index(drop=True); mols = [mols[i] for i in keep]; N = len(mols)
    E = np.zeros((N, 2048), np.float32)
    for i, m in enumerate(mols):
        DataStructs.ConvertToNumpyArray(AllChem.GetMorganFingerprintAsBitVect(m, 2, nBits=2048), E[i])
    D = np.nan_to_num(np.array([gen.process(s)[1:] for s in dfset.parent_smiles], dtype=np.float32))
    scaf = np.array([Chem.MolToSmiles(MurckoScaffold.GetScaffoldForMol(m)) for m in mols])
    return np.hstack([E, D]), dfset.pKi.values.astype(float), scaf
feat = {}
for n in sets:
    t0 = time.time(); feat[n] = featurize(sets[n]); print(f'{n} featurised: X{feat[n][0].shape}  [{time.time()-t0:.0f}s]')

In [ ]:
# ---- base models + leakage-safe scaffold split (as in the thesis) ----
def boost(): return XGBRegressor(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6, random_state=0, n_jobs=-1)
def rf():    return RandomForestRegressor(n_estimators=400, random_state=0, n_jobs=-1)
def scaffold_split(scaf, N, seed, tf=0.2):
    rng = np.random.RandomState(seed); g = {}
    for i, s in enumerate(scaf): g.setdefault(s, []).append(i)
    ks = list(g); rng.shuffle(ks); nt = int(round(tf*N)); te = []
    for k in ks:
        if len(te) >= nt: break
        te += g[k]
    te = set(te); return [i for i in range(N) if i not in te], sorted(te)
print('base pipeline ready')

## 1. Extra models + pKi distribution

In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl, xgboost as xgb
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from catboost import CatBoostRegressor, CatBoostClassifier
mpl.rcParams.update({"font.family": "serif", "font.size": 12, "savefig.dpi": 300, "savefig.bbox": "tight"})

def cbr():  return CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05, random_seed=0, verbose=0)
def bclf(): return XGBClassifier(n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.6, random_state=0, n_jobs=-1, eval_metric='logloss')
def rclf(): return RandomForestClassifier(n_estimators=400, random_state=0, n_jobs=-1)
def cclf(): return CatBoostClassifier(iterations=400, depth=6, learning_rate=0.05, random_seed=0, verbose=0)
CLF    = {'XGBoost': bclf, 'RandomForest': rclf, 'CatBoost': cclf}
THRESH = {'100 nM': 7.0, '1 uM': 6.0}
NSEED  = 12

fig, axes = plt.subplots(1, len(feat), figsize=(5.4*len(feat), 4.2))
if len(feat) == 1: axes = [axes]
for ax, t in zip(axes, feat):
    y = feat[t][1]
    ax.hist(y, bins=30, color='#6699cc', edgecolor='k', alpha=0.85)
    for lab, thr in THRESH.items():
        ax.axvline(thr, ls='--', lw=1.6, color='#c0392b' if lab == '100 nM' else '#e08e0b')
        ax.text(thr, ax.get_ylim()[1]*0.92, f' {lab}', rotation=90, va='top', ha='left', fontsize=9)
    ax.set_xlabel('pK$_i$'); ax.set_ylabel('compounds'); ax.set_title(t)
fig.tight_layout(); fig.savefig('pki_distribution.png'); plt.show()
print('saved pki_distribution.png  |  active-rate @100nM:', {t: round(float((feat[t][1] >= 7.0).mean()), 2) for t in feat})

In [ ]:
# ---- CatBoost regression, added to the scaffold-split comparison (for Table 3.7) ----
print('Regression, scaffold split, mean R2 over %d seeds (CatBoost):' % NSEED)
for t in feat:
    X, y, scaf = feat[t]; rs = []
    for s in range(NSEED):
        tr, te = scaffold_split(scaf, len(y), s)
        if len(te) < 5 or len(np.unique(y[te])) < 2: continue
        rs.append(r2_score(y[te], cbr().fit(X[tr], y[tr]).predict(X[te])))
    rs = np.array(rs)
    print(f'   {t:12s} CatBoost  R2 = {rs.mean():.3f} +/- {rs.std():.3f}')

## 2. Classification (active/inactive) + enrichment factor

In [ ]:
def enrichment(ybin, scores, frac):
    n = len(scores); k = max(1, int(round(frac*n)))
    idx = np.argsort(scores)[::-1][:k]; base = ybin.mean()
    return (ybin[idx].mean()/base) if base > 0 else np.nan

clf_rows = []
for lab, thr in THRESH.items():
    print(f'\n=== classification @ {lab} (active = pKi >= {thr}) ===')
    for t in feat:
        X, y, scaf = feat[t]; yb = (y >= thr).astype(int)
        for mn, mk in CLF.items():
            au, pr, e1, e5, e10 = [], [], [], [], []
            for s in range(NSEED):
                tr, te = scaffold_split(scaf, len(y), s)
                if len(np.unique(yb[tr])) < 2 or len(np.unique(yb[te])) < 2: continue
                p = mk().fit(X[tr], yb[tr]).predict_proba(X[te])[:, 1]
                au.append(roc_auc_score(yb[te], p)); pr.append(average_precision_score(yb[te], p))
                e1.append(enrichment(yb[te], p, 0.01)); e5.append(enrichment(yb[te], p, 0.05)); e10.append(enrichment(yb[te], p, 0.10))
            clf_rows.append((lab, t, mn, np.mean(au), np.mean(pr), np.nanmean(e1), np.nanmean(e5), np.nanmean(e10)))
            print(f'   {t:12s} {mn:12s} AUC={np.mean(au):.3f}  PR={np.mean(pr):.3f}  EF@1%={np.nanmean(e1):.2f}  EF@5%={np.nanmean(e5):.2f}  EF@10%={np.nanmean(e10):.2f}')
pd.DataFrame(clf_rows, columns=['threshold','target','model','ROC_AUC','PR_AUC','EF@1%','EF@5%','EF@10%']).to_csv('classification_metrics.csv', index=False)
print('\nsaved classification_metrics.csv')

In [ ]:
# ---- ROC + enrichment curve + predicted-value distribution (XGB, 100 nM, seed 0) ----
fig, axs = plt.subplots(1, 3, figsize=(15, 4.4))
colors = {'5-HT2A': '#c0392b', 'Dopamine D3': '#2c7fb8'}
for t in feat:
    X, y, scaf = feat[t]; yb = (y >= 7.0).astype(int); tr, te = scaffold_split(scaf, len(y), 0); col = colors.get(t, None)
    p = bclf().fit(X[tr], yb[tr]).predict_proba(X[te])[:, 1]
    fpr, tpr, _ = roc_curve(yb[te], p)
    axs[0].plot(fpr, tpr, color=col, lw=2, label=f'{t} (AUC {roc_auc_score(yb[te], p):.2f})')
    order = np.argsort(p)[::-1]; rec = np.cumsum(yb[te][order]) / yb[te].sum(); frac = np.arange(1, len(order)+1) / len(order)
    axs[1].plot(frac, rec, color=col, lw=2, label=t)
    pr = boost().fit(X[tr], y[tr]).predict(X[te]); axs[2].hist(pr, bins=25, histtype='step', lw=2, color=col, label=f'{t} predicted')
axs[0].plot([0, 1], [0, 1], 'k--', lw=1); axs[0].set_title('ROC (100 nM)'); axs[0].set_xlabel('false positive rate'); axs[0].set_ylabel('true positive rate'); axs[0].legend(fontsize=9)
axs[1].plot([0, 1], [0, 1], 'k--', lw=1); axs[1].set_title('Enrichment (actives recovered)'); axs[1].set_xlabel('fraction of library screened'); axs[1].set_ylabel('fraction of actives found'); axs[1].legend(fontsize=9)
axs[2].set_title('Predicted pK$_i$ distribution'); axs[2].set_xlabel('predicted pK$_i$'); axs[2].set_ylabel('compounds'); axs[2].legend(fontsize=9)
fig.tight_layout(); fig.savefig('enrichment_roc.png'); plt.show()
print('saved enrichment_roc.png')

## 3. SHAP feature importance (native XGBoost TreeSHAP)

In [ ]:
try:
    dcols = [c[0] for c in rdNormalizedDescriptors.RDKit2DNormalized().columns]
except Exception:
    dcols = []
fig, axes = plt.subplots(1, len(feat), figsize=(6.2*len(feat), 5))
if len(feat) == 1: axes = [axes]
for ax, t in zip(axes, feat):
    X, y, scaf = feat[t]; nd = X.shape[1] - 2048
    dd = list(dcols)
    if len(dd) == nd + 1: dd = dd[1:]
    if len(dd) != nd:     dd = [f'desc_{i}' for i in range(nd)]
    names = np.array([f'ECFP_{i}' for i in range(2048)] + dd)
    m = boost().fit(X, y)
    samp = X[np.random.RandomState(0).choice(len(X), min(1000, len(X)), replace=False)]
    contribs = m.get_booster().predict(xgb.DMatrix(samp), pred_contribs=True)
    mabs = np.abs(contribs[:, :-1]).mean(0)
    desc_idx = np.arange(2048, 2048 + nd)
    top = desc_idx[np.argsort(mabs[desc_idx])[::-1]][:15][::-1]
    ax.barh(range(len(top)), mabs[top], color='#3b7dd8')
    ax.set_yticks(range(len(top))); ax.set_yticklabels(names[top], fontsize=9)
    ax.set_xlabel('mean |SHAP|'); ax.set_title(f'{t}: top descriptors')
fig.tight_layout(); fig.savefig('shap_descriptors.png'); plt.show()
print('saved shap_descriptors.png')

## 4. Ablation: retrain WITHOUT promiscuous compounds

In [ ]:
PROM_CUT = 5
_locked = pd.read_csv(LOCKED)
_tc = _locked.groupby('molecule_chembl_id').target.nunique()
promis = set(_tc[_tc >= PROM_CUT].index)
print(f'promiscuous set: {len(promis)} molecules with >= {PROM_CUT} distinct targets\n')
print('Regression R2 (mean over %d seeds): full vs without-promiscuous' % NSEED)
for t in feat:
    X, y, scaf = feat[t]
    vm = np.array([Chem.MolFromSmiles(str(s)) is not None for s in sets[t].parent_smiles])
    ids = sets[t].molecule_chembl_id.values[vm]
    assert len(ids) == len(y), 'row alignment mismatch -- re-run the featurise cell'
    keep = ~np.isin(ids, list(promis)); nrem = int((~keep).sum())
    def _mean_r2(Xx, yy, ss):
        rs = []
        for s in range(NSEED):
            tr, te = scaffold_split(ss, len(yy), s)
            if len(te) < 5 or len(np.unique(yy[te])) < 2: continue
            rs.append(r2_score(yy[te], boost().fit(Xx[tr], yy[tr]).predict(Xx[te])))
        return float(np.mean(rs))
    full = _mean_r2(X, y, scaf); abl = _mean_r2(X[keep], y[keep], scaf[keep])
    print(f'   {t:12s} removed {nrem:3d}/{len(y)}   full={full:.3f}   without-promiscuous={abl:.3f}   d={abl-full:+.3f}')
print('\nDONE. Figures: pki_distribution.png, enrichment_roc.png, shap_descriptors.png | Table: classification_metrics.csv')